In [0]:
from pyspark.sql import functions as F

SILVER_TRAIN = "dbr_dev_ua5816bd.gulidov_artem_silver.silver_train"
SILVER_VALIDATION = "dbr_dev_ua5816bd.gulidov_artem_silver.silver_validation"

SILVER_DOCUMENTS = "dbr_dev_ua5816bd.gulidov_artem_silver.silver_documents"

documents = (
    spark.table(SILVER_TRAIN)
    .select("title", "context")
    .unionByName(
        spark.table(SILVER_VALIDATION)
        .select("title", "context")
    )
    .dropDuplicates(["title", "context"])
)

documents = (
    documents
    .withColumn(
        "document_id",
        F.sha2(
            F.concat_ws("||", F.col("title"), F.col("context")),
            256
        )
    )
    .withColumn("source", F.lit("SQuAD v2"))
    .withColumn("_processed_at", F.current_timestamp())
)

(
    documents.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_DOCUMENTS)
)

print(f"Unique documents: {documents.count():,}")